# Geolocalización de direcciones — Cali

Normaliza direcciones colombianas y obtiene coordenadas vía ArcGIS (fallback: Nominatim).
El procesamiento pesado corre con `geolocalizacion.py` desde terminal.

## Setup

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../src').resolve()))

import pandas as pd
from tqdm import tqdm
tqdm.pandas()

from geolocalizacion import normalizar_direccion, geocode_con_fallback

PATH_DATA = '../data/EMCALI/'
PATH_OUT  = '../data/Resultados/'

## Lectura de datos

In [3]:
df = pd.read_csv(PATH_DATA + 'info_usuarios.csv')
df = df[['producto', 'direccion_oficial', 'direccion_de_instalacion']]
print(df.shape)
df.head()


(909940, 3)


,producto,direccion_oficial,direccion_de_instalacion
0,1117913.0,CR 27 A 44-50,K 27A 44 50
1,1117918.0,CR 27 A 44-53,K 27A 44 53
2,1117923.0,CR 27 A 44-56,K 27A 44 56
3,1117928.0,CR 27 A 44-59,K 27A 44 59
4,1117933.0,CR 27 A 44-62,K 27A 44 62


## Normalización de direcciones

In [4]:
df[['direccion_limpia', 'fuente']] = df.progress_apply(geocode_con_fallback, axis=1)

print('Sin dirección:', df['direccion_limpia'].isna().sum())
df[['producto', 'direccion_oficial', 'direccion_limpia', 'fuente']].head(10)

100%|██████████| 909940/909940 [01:15<00:00, 12071.31it/s]


Sin dirección: 0


,producto,direccion_oficial,direccion_limpia,fuente
0,1117913.0,CR 27 A 44-50,"CARRERA 27 A # 44-50, Cali, Colombia",direccion_oficial
1,1117918.0,CR 27 A 44-53,"CARRERA 27 A # 44-53, Cali, Colombia",direccion_oficial
2,1117923.0,CR 27 A 44-56,"CARRERA 27 A # 44-56, Cali, Colombia",direccion_oficial
3,1117928.0,CR 27 A 44-59,"CARRERA 27 A # 44-59, Cali, Colombia",direccion_oficial
4,1117933.0,CR 27 A 44-62,"CARRERA 27 A # 44-62, Cali, Colombia",direccion_oficial
5,1117938.0,CR 27 A 44-67,"CARRERA 27 A # 44-67, Cali, Colombia",direccion_oficial
6,1117943.0,CR 27 A 44-68,"CARRERA 27 A # 44-68, Cali, Colombia",direccion_oficial
7,1117948.0,CR 27 A 49-02,"CARRERA 27 A # 49-2, Cali, Colombia",direccion_oficial
8,1117953.0,CR 27 A 49-05,"CARRERA 27 A # 49-5, Cali, Colombia",direccion_oficial
9,1117958.0,CR 27 A 49-08,"CARRERA 27 A # 49-8, Cali, Colombia",direccion_oficial


In [ ]:
# Deduplicar por dirección limpia antes de geocodificar
# (ahorra llamadas a la API si muchos productos comparten dirección)
df_unique = df.drop_duplicates(subset='direccion_limpia').copy()
df_unique[['lat', 'lon', 'source']] = None
print(f'Direcciones únicas a geocodificar: {len(df_unique):,}')

df_unique.to_csv(PATH_OUT + 'direcciones_a_geocodificar.csv', index=False)
print('Guardado en direcciones_a_geocodificar.csv')


## Geocodificación

Correr desde terminal (soporta interrupción y reanudación):

```bash
python src/geolocalizacion.py \
    --input  data/Resultados/direcciones_a_geocodificar.csv \
    --output data/Resultados/georeferencias.csv \
    --chunk_size 5000 \
    --workers 4
```

## Consolidar resultados

In [ ]:
georef = pd.read_csv(PATH_OUT + 'georeferencias.csv')
georef = georef[['direccion_limpia', 'lat', 'lon', 'source']]

print('Geocodificadas:', georef['lat'].notna().sum(), '/', len(georef))
print('Por fuente:')
print(georef['source'].value_counts())


In [ ]:
# Pegar coordenadas de vuelta al dataset original
df_final = df.merge(georef[['direccion_limpia', 'lat', 'lon', 'source']],
                    on='direccion_limpia', how='left')

print(f'Sin coordenadas: {df_final["lat"].isna().sum():,}')
df_final.to_csv(PATH_OUT + 'georeferencias_productos.csv', index=False)
print('Guardado en georeferencias_productos.csv')
